# Data and Vocabulary Preparation

In [1]:
# Download and decompress text8
# !wget http://mattmahoney.net/dc/text8.zip -O text8.gz
# !gzip -d text8.gz -f

In [2]:
from collections import Counter

# Load and preprocess the corpus
with open("text8", "r") as f:
    text = f.read().lower()
    tokens = text.split()
    
min_freq = 5
word_freq = Counter(tokens) # gets the freq of all unique words
vocab = [(w, c) for w, c in word_freq.items() if c >= min_freq]

vocab.sort(key=lambda x : (-x[1], x[0]))

# Add <UNK> to the vocab. Its frequency is the sum of all rare words.
unk_count = sum(c for w, c in word_freq.items() if c < min_freq)
vocab.append(('<UNK>', unk_count))

vocab_size = len(vocab)

In [3]:
word_to_idx = {w: i for i, (w,_) in enumerate(vocab)}
idx_to_word = {i: w for i, (w,_) in enumerate(vocab)}

In [4]:
print(f"Original token count: {len(tokens):,}")
print(f"Filtered vocabulary size: {len(vocab):,}")

Original token count: 17,005,207
Filtered vocabulary size: 71,291


In [5]:
unk_idx = word_to_idx['<UNK>']
train_data = []

for token in tokens:
    # Get the ID. If it's not in our filtered vocab, use <UNK>
    token_id = word_to_idx.get(token, unk_idx)
    train_data.append(token_id)

print(f"Full corpus ID list created. Length: {len(train_data):,}")

Full corpus ID list created. Length: 17,005,207


# SkipGram Training Pair Generation

In [6]:
import torch
from torch.utils.data import Dataset, DataLoader
import random

class SkipGramDataset(Dataset):
    """
    A PyTorch Dataset for generating skip-gram (center_word, context_word) pairs.
    """
    def __init__(self, token_ids, window_size):
        """
        Args:
            token_ids (list[int]): The list of token IDs.
            window_size (int): The maximum window size.
        """
        self.token_ids   = token_ids
        self.window_size = window_size
        
        # This will store all our (center, context) training pairs
        self.pairs = []
        
        print("Generating training pairs...")
        self._generate_pairs()
        print(f"Generated {len(self.pairs):,} training pairs.")
        
    def _generate_pairs(self):
        """
        Generates all (center, context) pairs and stores them in self.pairs.
        """
        num_tokens = len(self.token_ids)
        
        # Iterate over each token in the corpus to be a center word
        for center_word_idx in range(num_tokens):
            center_word_id = self.token_ids[center_word_idx]
            # Pick a random window size
            current_window_size = random.randint(1, self.window_size)
            
            start_idx = max(0, center_word_idx - current_window_size)
            end_idx   = min(num_tokens, center_word_idx + current_window_size + 1)
            
            for context_word_idx in range(start_idx, end_idx):
                if center_word_idx == context_word_idx:
                    continue
                context_word_id = self.token_ids[context_word_idx]
                self.pairs.append((center_word_id, context_word_id))
    
    def __len__(self):
        """Returns the total number of training pairs."""
        return len(self.pairs)
    
    def __getitem__(self, idx):
        """
        Returns a single (center_word, context_word) pair using tensor.
        """
        center_id, context_id = self.pairs[idx]
        
        # Return as tensors
        return (
            torch.tensor(center_id, dtype=torch.long), 
            torch.tensor(context_id, dtype=torch.long)
        )

In [7]:
window_size = 5
batch_size  = 1024

test_token_ids = train_data[:100000]
skipgram_dataset = SkipGramDataset(test_token_ids, window_size)

dataloader = DataLoader(
    skipgram_dataset, 
    batch_size=batch_size, 
    shuffle=True, 
    num_workers=2 
)

print("\n--- Testing the DataLoader ---")
center_batch, context_batch = next(iter(dataloader))

print(f"Center batch shape:  {center_batch.shape}")
print(f"Context batch shape: {context_batch.shape}")

print("\nExample center IDs:")
print(center_batch[:5])

print("\nExample context IDs:")
print(context_batch[:5])

Generating training pairs...
Generated 599,754 training pairs.

--- Testing the DataLoader ---
Center batch shape:  torch.Size([1024])
Context batch shape: torch.Size([1024])

Example center IDs:
tensor([   2, 1270,  128,  465,  978])

Example context IDs:
tensor([29, 21, 76, 65, 10])


# Naive SkipGram Model(Full Softmax)

In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import functional as F

n_embd = 300

class SkipGramModel(nn.Module):
    """
    Implements the Naive Skip-Gram model with Full Softmax.
    """
    def __init__(self):
        super().__init__()
        self.vocab_size = vocab_size
        self.n_embd = n_embd
        
        self.input_embeddings  = nn.Embedding(vocab_size, n_embd)  # Center/input word embeddings (vocab_size, n_embd)
        self.output_embeddings = nn.Embedding(vocab_size, n_embd)  # Context/output word embeddings (vocab_size, n_embd)
        
        self.input_embeddings.weight.data.uniform_(-0.5 , 0.5)
        self.output_embeddings.weight.data.uniform_(-0.5 , 0.5)
        
    def forward(self, center_words, context_words=None):
        """
        Forward pass to get the center embeddings.
        center_words: tensor of shape (batch_size,)
        """
        center_embeds = self.input_embeddings(center_words) # (batch_size, n_embd)
        logits = center_embeds @ self.output_embeddings.weight.T # (batch_size, n_embd) @ (vocab_size, n_embd).T ---> (batch_size, vocab_size)
        loss = None
        if context_words is not None:
            loss = F.cross_entropy(logits, context_words) # does softmax first then cross entropy
        return logits, loss

In [ ]:
lr = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

model = SkipGramModel()
m = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr)

cuda


In [ ]:
import matplotlib.pyplot as plt

# Use FULL dataset
skipgram_dataset = SkipGramDataset(train_data, window_size)  # Remove [:100000]
dataloader = DataLoader(
    skipgram_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2
)

num_epochs = 3  # Train for 3 full passes through the data
print_every = 100  # Print loss every 100 batches

running_loss = 0.0
losses = []
grad_norms = []
steps = []
global_step = 0

for epoch in range(num_epochs):
    print(f"\n{'='*50}")
    print(f"Starting Epoch {epoch+1}/{num_epochs}")
    print(f"{'='*50}")
    
    for i, (center_ids, context_ids) in enumerate(dataloader):
        center_ids = center_ids.to(device)
        context_ids = context_ids.to(device)
        
        logits, loss = model(center_ids, context_ids)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        global_step += 1
        
        # Log metrics every print_every steps
        if (i + 1) % print_every == 0:
            # Compute gradient norm
            total_norm = 0
            for p in model.parameters():
                if p.grad is not None:
                    total_norm += p.grad.data.norm(2).item() ** 2
            total_norm = total_norm ** 0.5
            
            avg_loss = running_loss / print_every
            losses.append(avg_loss)
            grad_norms.append(total_norm)
            steps.append(global_step)
            
            print(f"Epoch {epoch+1}, Batch {i+1}/{len(dataloader)}, "
                  f"Loss: {avg_loss:.4f}, Grad Norm: {total_norm:.4f}")
            
            running_loss = 0.0
    
    print(f"Completed Epoch {epoch+1}/{num_epochs}")

print("\nTraining Complete!")
print(f"Total batches processed: {global_step}")

# --- Plotting ---
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(steps, losses, label="Loss", color='blue', linewidth=2)
plt.xlabel("Global Step", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.title("Training Loss Over Time", fontsize=14)
plt.ylim([min(losses) - 0.2, max(losses) + 0.2])
plt.grid(alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(steps, grad_norms, label="Grad Norm", color='orange', linewidth=2)
plt.xlabel("Global Step", fontsize=12)
plt.ylabel("Gradient Norm", fontsize=12)
plt.title("Gradient Norm Over Time", fontsize=14)
plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

Generating training pairs...
